In [ ]:
"""
sandbox_lite.ipynb

A sandbox to develope a lighter version of the code.

Author: Stellina X. Ao
Created: 2026-05-29
Last Modified: 2026-05-29
Python Version: 3.11.14
"""

from squiggs.neuron_viewer import NeuronViewer
from sg.fitter import LVMFamily

import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""
TODO
1. replicate liska
    - why is liska better than ridgecv 
        - try fitting the splines first
2. RidgeCV-fy
"""

## lite

In [ ]:
# make sure that encoding weights make sense
# task responsive percentage (~50)
# replicate the baseline v task var plot
# cvr2 and delta r2
# add movement (average normalized me on a trial)
# add time
# one regressor

In [ ]:
from core.data import load_sess

# get data
(spike_times, trial_data, psths, session_data, regions) = load_sess(
    subj_id=subj_id,
    sess_id=sess_id,
    tpre=0.5,
    tpost=1,
    alignment="choice",
    tpre_ref=0.5,
    tpost_ref=1,
    alignment_ref="choice",
    binwidth_ms=25,
    thresh=1,
)

In [ ]:
from squiggs.renderers import PETHRasterRenderer
from core.data import get_psths_cond, get_choice_ts
from utils.paths import FIGURES_DIR
from pathlib import Path

reg = "DLS"
mode = "response"

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(trial_data, mode=mode),
    spike_times=spike_times[reg],
    peths=get_psths_cond(psths[reg], trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode,
)

nv = NeuronViewer(
    num_units=psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

In [ ]:
from sg.fitlvm_utils import get_data_model
import numpy as np

# make robs
# when constructing the design matrix, add the idx as a drift term
(data_gd, train_dl, val_dl, test_dl, indices, num_trials, num_tv, num_units) = (
    get_data_model(
        psths,
        trial_data,
        strategy_filter=None,
        regions=regions,
        norm=True,
        num_tents=5,
        task_vars=[
            "response",
            "rewarded",
            "block_side",
            "response_prev",
            "rewarded_prev",
        ],
        sanity_check=0,
    )
)
sample = data_gd[:]
robs = sample["robs"].detach().cpu().numpy()
tvs = sample["tv"].detach().cpu().numpy()
tents = sample["tents"].detach().cpu().numpy()

In [ ]:
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import r2_score

"""
- drift: train & test
- regularization for baseline
"""

# FIT BASELINE AND TASKVAR TO TRAIN
# ridge may not be correct for baseline

n_samples = robs.shape[0]
train_idxs = np.sort(np.random.choice(n_samples, int(n_samples * 0.8), replace=False))
test_idxs = np.setdiff1d(np.arange(n_samples), train_idxs)


def get_baseline(tents, robs):
    lr = LinearRegression().fit(tents, robs)
    drift = lr.predict(tents)
    return drift


def subtract_baseline(tents, robs):
    return robs - get_baseline(tents, robs)


n_folds = 20
scores_cv = np.zeros((n_folds, num_units))
# for i, (train_idxs, test_idxs) in enumerate(KFold(n_splits=n_folds).split(robs_bs)):

p_train = 0.8
n_samples = robs.shape[0]

for i in range(n_folds):
    train_idxs = np.sort(
        np.random.choice(n_samples, int(n_samples * p_train), replace=False)
    )
    test_idxs = np.setdiff1d(np.arange(n_samples), train_idxs)

    # have to regularize the baseline fit
    # and have to _fit_ it, not leave it as a preprocessing step
    # ridge = RidgeCV(alphas=np.logspace(-5, 5, 11, base=10), alpha_per_target=True).fit(tents[train_idxs], robs[train_idxs])
    ridge = RidgeCV(alphas=np.logspace(-5, 5, 11, base=10), alpha_per_target=True).fit(
        tents[train_idxs], robs[train_idxs]
    )
    robs_baseline_train = ridge.predict(tents[train_idxs])

    robs_scores = r2_score(
        robs[test_idxs], ridge.predict(tents[test_idxs]), multioutput="raw_values"
    )
    scores_cv[i] = robs_scores

    # rcv_taskvar = RidgeCV(
    #     alphas=np.logspace(-5, 5, 11, base=10),
    #     alpha_per_target=True,
    # ).fit(tvs[train_idxs], robs[train_idxs]-robs_baseline_train)

    # robs_scores = r2_score(robs[test_idxs], ridge.predict(tents[test_idxs])+rcv_taskvar.predict(tvs[test_idxs]), multioutput="raw_values")
    # scores_cv[i] = robs_scores

scores = np.median(scores_cv, axis=0)

# PREDICT
robs_predict = ridge.predict(tents)
# robs_predict = get_baseline(tents, robs) + rcv_taskvar.predict(tvs)

In [ ]:
n_folds = 10
n_samples = robs.shape[0]
p_train = 0.8

dm = np.hstack((tents, tvs))

scores_cv = np.zeros((n_folds, num_units))

for i in range(n_folds):
    train_idxs = np.sort(
        np.random.choice(n_samples, int(n_samples * p_train), replace=False)
    )
    test_idxs = np.setdiff1d(np.arange(n_samples), train_idxs)

    rcv = RidgeCV(
        alphas=np.logspace(-5, 5, 11, base=10),
        alpha_per_target=True,
    ).fit(dm[train_idxs], robs[train_idxs])

    robs_scores = r2_score(
        robs[test_idxs], rcv.predict(dm[test_idxs]), multioutput="raw_values"
    )
    scores_cv[i] = robs_scores

scores = np.median(scores_cv, axis=0)
robs_predict = rcv.predict(dm)

In [ ]:
plt.figure()
plt.imshow(scores_cv, aspect="auto")
plt.show()

## liska

In [ ]:
# subsample for an even number of mb and mf trials

scores_liska_cv = np.zeros((n_folds, num_units))

for i in range(n_folds):
    family = LVMFamily(
        subj_id=subj_id,
        sess_id=sess_id,
        n_latents_mult=1,
        n_latents_addt=1,
        sanity_check=0,
        regions=None,
        refit=False,
        balance_strategy=False,
        task_vars=[
            "response",
            "rewarded",
            "block_side",
            "response_prev",
            "rewarded_prev",
        ],
        n_splines=5,
        tpre=0.5,
        tpost=1,
        binwidth_ms=25,
        alignment="choice",
        tv_reg={"l2": 0.1},
        seed=i,
    )

    family.fit_all(fit_lvms=False, update_cids=False)
    family.eval()

    scores_liska_cv[i] = family.res_baseline["r2test"]

scores_liska = np.median(scores_liska_cv, axis=0)

In [ ]:
family.res_baseline["r2test"]

In [ ]:
reg = "DLS"
mode = "response"

renderer = PETHRasterRenderer(
    event_times=get_choice_ts(family.trial_data, mode=mode),
    spike_times=family.spike_times[reg],
    peths=get_psths_cond(family.psths[reg], family.trial_data, mode=mode),
    pres=0.5,
    posts=1,
    binwidth_s=25 / 1000,
    s=0.2,
    linewidths=0.2,
    save_subdir=Path("peths") / subj_id / sess_id / reg / mode,
)

nv = NeuronViewer(
    num_units=family.psths[reg].shape[0], render_func=renderer, fig_dir=FIGURES_DIR
)

## compare

In [ ]:
plt.figure()
plt.hist(scores_liska)
plt.show()

In [ ]:
max(scores)

In [ ]:
plt.figure()
plt.hist(scores)
plt.show()

In [ ]:
idx = 15
print(scores_liska_cv[:, idx])
print(scores_cv[:, idx])

In [ ]:
# plot fit renderer on test set only

In [ ]:
test_dfs = family.test_dl.dataset[:]["dfs"].detach().cpu().numpy()
robs_unit = np.where(test_dfs[:, 1], robs[:, 1], 0)
rhat_unit = np.where(test_dfs[:, 1], family.res_taskvar["rhat"][:, 1], 0)

In [ ]:
unit_idx = 8
test_idxs_liska = np.where(test_dfs[:, unit_idx])[0]
robs_liska = robs[test_idxs_liska, unit_idx]
rhat_liska = family.res_taskvar["rhat"][test_idxs_liska, unit_idx]

plt.figure(tight_layout=True)
plt.plot(robs_liska)
plt.plot(rhat_liska)
plt.show()

In [ ]:
def get_r2(robs, rhat):
    ybar = robs.mean(axis=0)  # the average y value
    resids = robs - rhat  # the difference between observed and predicted
    residnull = robs - ybar  # the difference between observed and observed avg
    sstot = np.sum(residnull**2, axis=0)  # denom
    ssres = np.sum(resids**2, axis=0)  # num
    r2 = 1 - ssres / sstot

    return r2


get_r2(robs_liska, rhat_liska)

In [ ]:
plt.figure(tight_layout=True)

robs_lite = robs[test_idxs, unit_idx]
rhat_lite = robs_predict[test_idxs, unit_idx]
plt.plot(robs_lite)
plt.plot(rhat_lite)
plt.show()

In [ ]:
get_r2(robs_lite, rhat_lite)

In [ ]:
from squiggs.renderers import FitRendererCompare

renderer = FitRendererCompare(
    y=robs,
    yhat1=robs_predict,
    yhat2=family.mod_baseline(family.test_dl.dataset[:]).detach().numpy(),
    label1="lite",
    label2="liska",
    rsquared1=scores,
    rsquared2=scores_liska,
    # dfs2=family.test_dl.dataset[:]["dfs"][:].detach().cpu().numpy(),
)

nv = NeuronViewer(
    num_units=robs.shape[0],
    render_func=renderer,
    fig_dir=FIGURES_DIR,
)

In [ ]:
lr = LinearRegression().fit(scores.reshape(-1, 1), scores_liska)

plt.figure(tight_layout=True)
plt.scatter(scores, scores_liska)
# plt.scatter(renderer.rsquared1, renderer.rsquared2, s=0.5, alpha=0.5)
# plt.scatter(scores, renderer.rsquared2, s=0.5, alpha=0.5)
plt.plot([-0.5, 1], [-0.5, 1], color="#666666", label="unity")
plt.plot(
    [-0.5, 1],
    lr.predict(np.array([-0.5, 1]).reshape(-1, 1)),
    label=f"y={lr.coef_[0]:.3f}x+{lr.intercept_:.3f}",
)
plt.axvline(x=0, color="#666666")

plt.xlim([-0.5, 1])
plt.xlabel("lite")
plt.ylabel("liska")
plt.legend()
plt.show()

In [ ]:
np.round(np.where(scores > scores_liska)[0].shape[0] / len(scores), 3)

In [ ]:
plt.figure()
plt.hist(
    renderer.rsquared1,
    bins=np.linspace(-0.5, 1, 16),
    alpha=0.5,
    histtype="step",
    label="lite",
)
plt.hist(
    renderer.rsquared2,
    bins=np.linspace(-0.5, 1, 16),
    alpha=0.5,
    histtype="step",
    label="liska",
)
plt.axvline(x=0, color="k")
plt.legend()
plt.show()